# Yahoo Information

In [1]:
!pip3 install yfinance

  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
     ---------------------------------------- 0.0/3.0 MB ? eta -:--:--
     --- ------------------------------------ 0.3/3.0 MB 5.9 MB/s eta 0:00:01
     ----------------- ---------------------- 1.3/3.0 MB 13.7 MB/s eta 0:00:01
     ------------------------------------ --- 2.7/3.0 MB 19.4 MB/s eta 0:00:01
     ---------------------------------------- 3.0/3.0 MB 19.3 MB/s eta 0:00:00
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata 


[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import yfinance as yf
import pandas as pd
import os

### Get unique tickers

In [17]:
stock_data_dir = "../Data/Stock_data"
tickers = []

files = os.listdir(stock_data_dir)

for file in files:
    if file.endswith(".xlsx") or file.endswith(".xls"):
        path = os.path.join(stock_data_dir, file)

        # Read only column A
        df = pd.read_excel(path, header=None)

        # Ticker is in cell A4  → row index 3, col index 0
        ticker = df.iloc[3, 0]

        # Clean ticker if needed (strip spaces etc.)
        ticker = str(ticker).strip()
        tickers.append(ticker)
unique_tickers = sorted(set(tickers))

unique_tickers


['ARG.TO',
 'AYA.TO',
 'CG.TO',
 'CGC.V',
 'CKG.V',
 'DMX.V',
 'DV.V',
 'EDG.V',
 'ELO.TO',
 'EQX',
 'ESK.V',
 'FAS.V',
 'FDY.TO',
 'FINX.V',
 'FNM.V',
 'FOM.TO',
 'FURY.TO',
 'FVL.TO',
 'FWZ.V']

In [18]:
tickers_df = pd.DataFrame({'tickers': unique_tickers})
tickers_df

,tickers
0,ARG.TO
1,AYA.TO
2,CG.TO
3,CGC.V
4,CKG.V
5,DMX.V
6,DV.V
7,EDG.V
8,ELO.TO
9,EQX


In [19]:
tickers_df.to_csv("../Data/tickers.csv", index=False)

### Getting Yahoo Info (Static)

In [20]:
fundamentals = []

for ticker in tickers_df["tickers"]:
    t = yf.Ticker(ticker)
    info = t.info  # dict met heel veel keys (kan even duren)
    
    fundamentals.append({
        "ticker": ticker,
        "longName": info.get("longName"),
        "sector": info.get("sector"),
        "industry": info.get("industry"),
        "country": info.get("country"),
        "marketCap": info.get("marketCap"),
        "beta": info.get("beta"),
        "dividendYield": info.get("dividendYield"),
        "trailingPE": info.get("trailingPE"),
        "forwardPE": info.get("forwardPE"),
        "priceToBook": info.get("priceToBook"),
        "enterpriseToEbitda": info.get("enterpriseToEbitda"),
        "totalDebt": info.get("totalDebt"),
        "totalCash": info.get("totalCash"),
        # voeg toe wat nuttig lijkt
    })

fundamentals_df = pd.DataFrame(fundamentals)
fundamentals_df

,ticker,longName,sector,industry,country,marketCap,beta,dividendYield,trailingPE,forwardPE,priceToBook,enterpriseToEbitda,totalDebt,totalCash
0,ARG.TO,Amerigo Resources Ltd.,Basic Materials,Copper,Canada,539380800,2.032,4.79,18.555553,11.517242,5.045317,8.446,7.263000e+06,28156000.0
1,AYA.TO,Aya Gold & Silver Inc.,Basic Materials,Silver,Canada,2076003072,1.238,NaN,None,20.605635,5.232475,64.700,1.131910e+08,129184000.0
2,CG.TO,Centerra Gold Inc.,Basic Materials,Gold,Canada,3494628864,1.226,1.62,7.629956,21.121950,1.800790,6.847,1.762200e+07,568227968.0
3,CGC.V,Canadian Gold Corp.,Basic Materials,Gold,Canada,113711752,0.583,NaN,None,NaN,60.000004,-26.814,0.000000e+00,3465181.0
4,CKG.V,Chesapeake Gold Corp.,Basic Materials,Other Precious Metals & Mining,Canada,162211504,1.687,NaN,None,NaN,1.077070,-41.711,0.000000e+00,12141000.0
5,DMX.V,District Metals Corp.,Basic Materials,Other Industrial Metals & Mining,Canada,167138592,2.140,NaN,None,NaN,8.620690,-50.214,0.000000e+00,9890155.0
6,DV.V,Dolly Varden Silver Corporation,Basic Materials,Silver,Canada,477707264,1.890,NaN,None,-86.666664,4.140127,-13.499,0.000000e+00,34478232.0
7,EDG.V,Endurance Gold Corporation,Basic Materials,Gold,Canada,45836972,1.703,NaN,Infinity,NaN,2.653061,-39.938,0.000000e+00,3178093.0
8,ELO.TO,Eloro Resources Ltd.,Basic Materials,Gold,Canada,152056992,1.474,NaN,None,-15.666666,2.000000,-29.310,6.405700e+04,13214790.0
9,EQX,Equinox Gold Corp.,Basic Materials,Gold,Canada,9998051328,1.987,NaN,127.399994,11.688073,1.790332,13.042,1.938922e+09,403222016.0


In [23]:
fundamentals_df["industry"].unique()

array(['Copper', 'Silver', 'Gold', 'Other Precious Metals & Mining',
       'Other Industrial Metals & Mining'], dtype=object)

# Bi-weekly Beta Coefficiënt

### Get Industry Benchmark

Since our dataset contains multiple different Industries:
['Copper', 'Silver', 'Gold', 'Other Precious Metals & Mining', 'Other Industrial Metals & Mining']

It is necesarry to find a benchmark for each of these industries when computing the beta to increase accuracy of the beta coeff. 

In [29]:
start = "2024-01-01"
end   = "2025-12-31"

In [28]:
gold_benchmark = "GDX"
silver_benchmark = "SIL"
copper_benchmark = "COPX"
precious_metals_benchmark = "PICK"
industrial_mining_benchmark = "PICK"

benchmark_tickers = [gold_benchmark, silver_benchmark, copper_benchmark, precious_metals_benchmark]

In [27]:
# Create a benchmark map
benchmark_map = {
    "Gold": gold_benchmark,
    "Silver": silver_benchmark,
    "Copper": copper_benchmark,
    "Other Precious Metals & Mining": precious_metals_benchmark,
    "Other Industrial Metals & Mining": industrial_mining_benchmark,
}

In [33]:
prices = yf.download(
    tickers=benchmark_tickers,
    start=start,
    end=end,
    auto_adjust=True
)

prices.head()

[*********************100%***********************]  4 of 4 completed


Price           Close                                        High             \
Ticker           COPX        GDX       PICK        SIL       COPX        GDX   
Date                                                                           
2024-01-02  36.379265  30.205389  40.893002  27.046282  36.741007  30.877499   
2024-01-03  36.134846  29.414673  40.273415  26.207485  36.281498  29.711191   
2024-01-04  36.056633  29.424557  40.149494  26.363541  36.213061  29.612351   
2024-01-05  36.056633  29.424557  40.044640  26.324526  36.604128  30.007711   
2024-01-08  36.301052  29.355370  40.130428  26.334280  36.359710  29.523397   

Price                                   Low                                   \
Ticker           PICK        SIL       COPX        GDX       PICK        SIL   
Date                                                                           
2024-01-02  41.169432  27.699762  36.281499  30.146086  40.635633  26.978008   
2024-01-03  40.464059  26.578115  35.411369  29.187342  39.806337  26.070936   
2024-01-04  40.321074  26.461074  35.743777  29.098386  40.006514  25.944143   
2024-01-05  40.444993  26.909732  35.841543  29.167573  39.911190  26.168471   
2024-01-08  40.187623  26.461074  35.636233  28.880938  39.472710  25.934389   

Price            Open                                   Volume            \
Ticker           COPX        GDX       PICK        SIL    COPX       GDX   
Date                                                                       
2024-01-02  36.359711  30.679821  40.854872  27.690009  725400  21612300   
2024-01-03  35.851319  29.562932  40.273415  26.539100  365400  33470600   
2024-01-04  35.909980  29.503628  40.073239  26.217238  232900  17862300   
2024-01-05  36.076187  29.464092  39.930255  26.392800  160200  19585800   
2024-01-08  35.929535  28.950127  39.663354  26.022170  168200  14185500   

Price                        
Ticker         PICK     SIL  
Date                         
2024-01-02  1164000  450200  
2024-01-03   190200  465100  
2024-01-04    54300  310600  
2024-01-05    57700  417800  
2024-01-08   209600  212500

load in bi_weekly data

In [1]:
# import findspark
import findspark
# initialize findspark with spark directory
findspark.init()
# import pyspark
import pyspark
# create spark context
sc = pyspark.SparkContext()
# create spark session
spark = pyspark.sql.SparkSession(sc)

In [2]:
spark

In [3]:
# import pyspark functionalities
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

# turn of warnings
import warnings
warnings.filterwarnings("ignore")

In [ ]:
df = spark.read.parquet("../Cleaned_data/monthly_partitioned_data.parquet")
df.show()

AnalysisException: [PATH_NOT_FOUND] Path does not exist: file:/c:/Users/renzo/OneDrive/Documenten/Academiejaar 2025 - 2026/Big_Data_Git/Big_data_group_2/Cleaned_data/monthly_partitioned_data.parquet. SQLSTATE: 42K03